# TruthShield 03: Robustness to social-media compression (in parts)
Re-encodes the **762 test videos** with H.264 at two quality levels and re-evaluates the
trained models from notebook 02 (no re-training). CRF 28 is typical of social-media uploads,
CRF 35 is heavily compressed.

Parts: one per (quality level × generator). Every video is saved as soon as it is done.
Needs notebooks 00–02 finished. T4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, glob
ROOT = '/content/drive/MyDrive/TruthShield'
%cd /content
if not os.path.exists('/content/TruthShield'):
    !git clone -q https://github.com/Jawad374/TruthShield.git /content/TruthShield
!git -C /content/TruthShield pull -q
%cd /content/TruthShield
!nvidia-smi --query-gpu=name,memory.total --format=csv

!pip install -q decord ffmpeg-python einops timm transformers
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
if not os.path.exists('/content/HAVIC'):
    !git clone -q --depth 1 https://github.com/tuffy-studio/HAVIC /content/HAVIC

In [ ]:
import zipfile, pandas as pd
LOCAL = '/content/hifi'
os.makedirs(LOCAL, exist_ok=True)
for z in sorted(glob.glob(f'{ROOT}/data/HiFi-AVDF/*.zip')):
    eng = os.path.basename(z)[:-4]
    if not os.path.exists(f'{LOCAL}/.done_{eng}'):
        print('unzipping', eng)
        with zipfile.ZipFile(z) as f:
            f.extractall(LOCAL)
        open(f'{LOCAL}/.done_{eng}', 'w').close()
SPLIT = f'{ROOT}/splits/hifi_main_abs.csv'
s = pd.read_csv(SPLIT); s = s[s.split == 'test']
print(len(s), 'test clips')

## Cache compressed test videos: one part per (quality × generator)

In [ ]:
CRFS = [28, 35]
ENGINES = ['kling2.5', 'pixverseV5', 'seedance1.0', 'sora2', 'veo3.1', 'wan2.5']
PARTS_THIS_SESSION = 12
def todo_parts():
    out = []
    for crf in CRFS:
        done = {os.path.basename(f) for f in glob.glob(f'{ROOT}/cache/hifi_crf{crf}/clips/*.pt')}
        for e in ENGINES:
            sub = s[s.engine == e]
            n = sum(p.replace('/', '__').replace('.mp4', '.pt') in done for p in sub.video_path)
            print(f'  crf{crf} {e:12s} {n:3d}/{len(sub)}', 'DONE' if n == len(sub) else '')
            if n < len(sub): out.append((crf, e))
    return out
parts = todo_parts()
for crf, e in parts[:PARTS_THIS_SESSION]:
    print(f'========== PART: CRF {crf} / {e} ==========')
    !python -u cache_features.py --split_csv {SPLIT} --havic_root /content/HAVIC --havic_weights {ROOT}/weights/best_ft_model_clean.pth --out_dir {ROOT}/cache/hifi_crf{crf} --only_split test --engines {e} --crf {crf}
parts = todo_parts()

## Merge parts and evaluate (run when all parts are DONE)

In [ ]:
for crf in CRFS:
    d = f'{ROOT}/cache/hifi_crf{crf}'
    !python -u cache_features.py --pack --out_dir {d}
    !python -u evaluate_benchmark.py --protocol robust --cache_dir {d} --eval_cache_dir {d} --splits_dir {ROOT}/splits --runs_dir {ROOT}/runs --results_dir {ROOT}/results --tag crf{crf} | sed -n '/^# Robust/,$p'